# Listwise Learning to Rank con AdaRank — consultas LOINC

Notebook de demostración. Toda la lógica está en `src/`:

| Script | Qué hace |
|---|---|
| `src/queries.py` | Las 15 consultas y la regla de relevancia 0–3 |
| `src/build_collection.py` | (requiere `Loinc.csv`) selecciona 12 consultas por estratificación (dominio × muestra) sobre `COMMON_TEST_RANK` y construye la colección por pooling → `data/collection.csv` |
| `src/label.py` | Aplica la regla a la colección (67 docs del Excel + pooling) → `data/relevance_labels.csv` |
| `src/build_dataset.py` | Features f1–f5 + LETOR en tres configuraciones → `data/train*.txt` |
| `src/evaluate.py` | Evaluación leave-one-query-out de AdaRank frente a baselines |

| Configuración | Archivo | Consultas × documentos |
|---|---|---|
| official | `data/train.txt` | 3 × 67 (entregable obligatorio) |
| terms | `data/train_terms.txt` | 3 × 1222 (más términos) |
| extended | `data/train_extended.txt` | 15 × 1222 (más términos y consultas) |

La implementación de AdaRank (`adarank.py`, `metrics.py`, `utils.py`) está adaptada de
[rueycheng/AdaRank](https://github.com/rueycheng/AdaRank).

In [ ]:
import sys
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.datasets import load_svmlight_file

from adarank import AdaRank
from metrics import NDCGScorer
import build_dataset
from build_dataset import FEATURES

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

## 1. Construir el training set

Regenera etiquetas, features y archivos LETOR, y ejecuta las comprobaciones
(201 pares oficiales, misma lista de documentos por consulta, etiquetas 0–3, features finitas,
34 casos de referencia, incluidos distractores con etiqueta 0).

In [ ]:
build_dataset.main()

In [ ]:
labels = pd.read_csv('data/relevance_labels.csv', dtype={'loinc_num': str})
official = labels[labels.official & (labels.doc_source == 'excel')]
official.pivot_table(index='query', columns='relevance', values='loinc_num',
                     aggfunc='count', fill_value=0, sort=False)

### Documentos del Excel con relevancia > 0 y su justificación

In [ ]:
(official[official.relevance > 0]
 .sort_values(['qid', 'relevance'], ascending=[True, False])
 [['query', 'loinc_num', 'long_common_name', 'relevance', 'rationale']])

### Colección completa: distribución de etiquetas de las 15 consultas (1222 documentos)

La colección se construye con `src/build_collection.py` (pooling sobre la tabla oficial de LOINC);
aquí se usa el resultado ya guardado en `data/collection.csv`.

In [ ]:
pd.read_csv('data/collection.csv').doc_source.value_counts()

In [ ]:
labels.pivot_table(index='query', columns='relevance', values='loinc_num',
                   aggfunc='count', fill_value=0, sort=False)

## 2. Features (calculadas sin mirar la etiqueta)

In [ ]:
feats = pd.read_csv('data/features.csv', dtype={'loinc_num': str})
data = {c: labels.merge(g.drop(columns=['config', 'query']), on=['qid', 'loinc_num'])
        for c, g in feats.groupby('config')}
data['official'][data['official'].qid == 3].sort_values('relevance', ascending=False)[
    ['long_common_name', 'relevance'] + FEATURES].head(8)

### Formato LETOR (`data/train.txt`)

In [ ]:
print(''.join(open('data/train.txt', encoding='utf-8').readlines()[:5]))

## 3. Entrenamiento y evaluación leave-one-query-out (oficial)

Con 3 consultas, cada fold entrena con 2 y evalúa en la tercera. No se dividen
documentos de una misma consulta entre train y test: la unidad Listwise es la lista.

In [ ]:
X, y, qid = load_svmlight_file('data/train.txt', query_id=True, n_features=len(FEATURES))
rows = []
for test_q in np.unique(qid):
    tr, te = qid != test_q, qid == test_q
    model = AdaRank(max_iter=100, estop=10, scorer=NDCGScorer(k=10)).fit(X[tr], y[tr], qid[tr])
    for name, pred in [('AdaRank', model.predict(X[te], qid[te])),
                       ('baseline f1 (TF-IDF)', X[te][:, 0].toarray().ravel())]:
        rows.append({'test_qid': int(test_q), 'sistema': name,
                     **{f'NDCG@{k}': NDCGScorer(k=k)(y[te], pred, qid[te])[0] for k in (1, 3, 5, 10)}})
loqo = pd.DataFrame(rows)
loqo

In [ ]:
loqo.groupby('sistema').mean(numeric_only=True).drop(columns='test_qid')

## 4. Modelo final (3 consultas oficiales) y ranking por consulta

In [ ]:
model = AdaRank(max_iter=100, estop=10, scorer=NDCGScorer(k=10)).fit(X, y, qid)
print(dict(zip(FEATURES, model.coef_.round(4))))

def rank(df, k=8):
    df = df.copy()
    df['score'] = model.predict(csr_matrix(df[FEATURES].to_numpy()), None)
    ndcg = NDCGScorer(k=10)(df.relevance.to_numpy(), df.score.to_numpy(), np.zeros(len(df)))[0]
    top = df.sort_values('score', ascending=False)[['long_common_name', 'doc_source', 'relevance', 'score']].head(k)
    return ndcg, top

for q, g in data['official'].groupby('query', sort=False):
    ndcg, top = rank(g)
    print(f'\n== {q}  (NDCG@10 = {ndcg:.3f})')
    print(top.to_string(index=False))

## 5. Consulta antigua con términos nuevos

El mismo modelo (entrenado con 3 × 67) ordena ahora la colección completa de 1222 documentos.
Como los documentos del pooling también están etiquetados con la regla, podemos medir el NDCG.

In [ ]:
for qid_ in (1, 3):
    g = data['terms'][data['terms'].qid == qid_]
    ndcg, top = rank(g)
    print(f"\n== {g['query'].iloc[0]}  (NDCG@10 = {ndcg:.3f})")
    print(top.to_string(index=False))

En *white blood cells count*, las proporciones *X cells/Leukocytes in Blood* contienen
literalmente "cells" y "blood" y desplazan al recuento total de leucocitos (relevancia 3): el pooling
genera **distractores difíciles** para features léxicas.

## 6. Consultas nuevas con términos nuevos

Tres consultas de la selección estratificada, de dominios distintos y no vistas en el entrenamiento:
`prothrombin time in plasma` (coagulación, qid 7), `SARS-CoV-2 RNA in respiratory specimen`
(microbiología, qid 9) y `red blood cells count` (hematología, qid 5). Se puntúan los **1222 documentos
únicos** de la colección (el notebook inicial repetía cada documento una vez por consulta).

In [ ]:
for qid_ in (7, 9, 5):
    g = data['extended'][data['extended'].qid == qid_]
    assert g.loinc_num.is_unique
    ndcg, top = rank(g)
    print(f"\n== {g['query'].iloc[0]}  (NDCG@10 = {ndcg:.3f})")
    print(top.to_string(index=False))

*red blood cells count* obtiene NDCG@10 = 0 y no es un error: los documentos *MCHC* y *MCV in Red Blood Cells*
contienen literalmente "red blood cells" (en LOINC, "Red Blood Cells" es una muestra), pero son índices de
hemoglobina y tienen relevancia 0. El recuento *Erythrocytes [#/volume] in Blood* no comparte esas palabras.
Es el límite de las features léxicas: ven palabras, no conceptos.

## 7. Evaluación leave-one-query-out en las tres configuraciones

Misma evaluación que `python src/evaluate.py`: escribe los resultados en `results/`, de donde salen las tablas del informe.

In [ ]:
import evaluate
for config in build_dataset.CONFIGS:
    evaluate.evaluate(config)